# MedDial on Colab: BigQuery in, GPU extraction out

The CSV distribution of MIMIC-III is tens of gigabytes, which a Colab runtime will not hold, and
reference extraction is bound by how fast a local model can read a discharge summary. This notebook
resolves both: MIMIC-III is read from `physionet-data` on BigQuery, and the extractor runs on the
runtime's GPU.

**Before you start you need all three of these.** Any one missing and the run stops at step 3.

1. PhysioNet credentialing for MIMIC-III, with the *Google BigQuery* access request approved for the
   same Google account you will sign into below (PhysioNet account settings -> Cloud).
2. A Google Cloud project of your own. BigQuery bills the account that runs the query, not the one
   that publishes the data. Reading the six tables costs a few GB against the monthly free tier.
3. A GPU runtime: **Runtime -> Change runtime type -> T4 / L4 / A100**.

**What leaves this runtime: nothing.** The model server runs inside the runtime and is reached over
loopback, which is what decision D2 / GOV-3 requires -- MIMIC-derived text is never sent to a hosted
API. Reading MIMIC-III *from* BigQuery is the same direction as downloading the CSVs and does not
touch that rule. The output written under `/content` is derived from restricted data: a Colab runtime
is ephemeral, so decide deliberately where it goes at the end, and do not mount Drive by reflex.


## 0. Confirm the GPU


In [ ]:
!nvidia-smi

# No output above means the runtime has no GPU: Runtime -> Change runtime type -> T4 / L4 / A100.
# Extraction will still run on CPU, at roughly an order of magnitude less throughput.


## 1. Serve a model on the GPU

Ollama is installed into the runtime and left listening on `localhost:11434`. The provider layer
refuses to send restricted clinical text anywhere that is not loopback, and checks that *before* it
opens a socket, so this is the only shape of model server the pipeline accepts.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os, subprocess, time
import httpx

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        httpx.get('http://localhost:11434/api/tags', timeout=2.0).raise_for_status()
        print('ollama is serving')
        break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError('ollama did not come up; re-run this cell')


### Pick the largest extractor the GPU actually holds

Implementation Plan 12.4 asks for the largest extractor the hardware allows, because extraction error
propagates into every downstream metric. What it must not do is exceed VRAM: a model that swaps is
not a slow run, it is a stalled one. The tag below is chosen from the card Colab handed you.


In [ ]:
# torch is preinstalled on Colab and is not a dependency of this package, so a
# runtime without it is assumed to have no GPU rather than failing here.
try:
    import torch
    gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
except ImportError:
    gib = 0

# Weights at Q4 plus KV cache, with headroom. Raise these only after watching nvidia-smi.
if gib >= 38:      EXTRACTOR = 'qwen2.5:32b'   # A100 40GB
elif gib >= 21:    EXTRACTOR = 'qwen2.5:14b'   # L4 24GB
elif gib >= 14:    EXTRACTOR = 'qwen2.5:14b'   # T4 16GB, tight but fits at Q4
else:              EXTRACTOR = 'qwen2.5:7b'    # CPU or a small card

os.environ['MEDDIAL_EXTRACTOR'] = EXTRACTOR
print(f'{gib:.0f} GiB of VRAM -> {EXTRACTOR}')


In [ ]:
!ollama pull $MEDDIAL_EXTRACTOR


## 2. Install MedDial

The `bigquery` extra adds the BigQuery client; without it the CSV path still works and `--bigquery`
fails with an instruction to install it.

What is deliberately *not* installed is the `eval` extra -- DeepEval, transformers,
sentence-transformers and the rest of the scoring stack. `meddial-cohort` and `meddial-scr` import
none of it, and installing it here would do more than waste several gigabytes: this image ships
`huggingface-hub` 1.x for its own `gradio` and `diffusers`, and pulling in `transformers` 4.x drags
the hub back to 0.x and breaks both. Add `[eval]` only when you get as far as scoring, and expect to
restart the runtime if you do.


In [ ]:
!git clone -q https://github.com/alongott15/FinalProject-MedDial.git /content/FinalProject-MedDial
%pip install -q -e '/content/FinalProject-MedDial[bigquery]'


## 3. Sign in, and prove the access works before spending an hour on it

Set `MIMIC_BIGQUERY_PROJECT` to your own project id. The pre-flight query below reads one small
table: if PhysioNet credentialing is not linked to this Google account, it fails here in a second
rather than part-way through building the cohort.


In [ ]:
from google.colab import auth

auth.authenticate_user()

os.environ['MIMIC_BIGQUERY_PROJECT'] = 'your-gcp-project-id'  # <- edit this


In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=os.environ['MIMIC_BIGQUERY_PROJECT'])
rows = client.query('SELECT COUNT(*) AS n FROM `physionet-data.mimiciii_clinical.icustays`').result()
print('ICU stays visible:', next(iter(rows)).n)


## 4. Build the cohort

Criteria E1-E10 are applied to the **structured** fields only -- admission type, length of stay,
ICD-9 code sets, a Quan et al. 2005 Charlson index -- never to note vocabulary. The exclusion count at
each criterion is printed and written to the manifest, so the flow diagram is a by-product of
selection rather than something reconstructed afterwards.

The manifest records a `bigquery-sha256:` snapshot over each table's id, row count and last
modification. That is the BigQuery equivalent of hashing the CSV bytes, and it is deliberately not
interchangeable with one: **the sample is salted with it**, so a cohort you built earlier from the
CSVs will not reproduce here, and `meddial-scr` will refuse to mix the two. Pick one backend and
build the whole pipeline on it.


In [ ]:
!meddial-cohort \
    --bigquery \
    --out /content/meddial-out/cohort \
    --n 200


## 5. Extract the references

One Structured Clinical Reference per selected admission, read off the admission's whole discharge
documentation. Extraction is resumable: a case whose file already exists is skipped, so when the
Colab runtime is recycled mid-run -- and on a long run it will be -- re-running this cell continues
where it stopped, provided the output directory survived. If it did not, copy it off first (step 6).


In [ ]:
!meddial-scr \
    --bigquery \
    --cohort /content/meddial-out/cohort/cohort_private_manifest.json \
    --out /content/meddial-out/references \
    --extractor $MEDDIAL_EXTRACTOR \
    --extractor-family qwen


## 6. Take the output with you

Everything under `/content/meddial-out` is derived from restricted data and disappears with the
runtime. Where it may be stored is a governance decision, not a convenience one, so this notebook
does not mount Drive or upload anything for you. The cell below only packages the directory; run it,
then move the archive somewhere your data use agreement covers.


In [ ]:
!cd /content && zip -qr meddial-out.zip meddial-out && ls -lh /content/meddial-out.zip

# from google.colab import files; files.download('/content/meddial-out.zip')
